# SILVA Method Adaptation Atlas

This notebook is the executable companion to the method adaptation atlas. It
uses only `silva_networks` package APIs and cites external sources as lineage,
as package-native derivations and executable examples.

The common contract is

$$
z^\star=f_\theta(z^\star,x),\qquad r=f_\theta(z^\star,x)-z^\star.
$$

Sources to cite as needed: Deep Implicit Layers, Neural ODEs, DEQ, MDEQ,
Jacobian-regularized DEQs, TorchDEQ, OptNet, differentiable convex
optimization layers, RAFT, and DEQ-Flow.

SILVA Networks software archive: https://doi.org/10.5281/zenodo.21770098

<!-- silva-numbered-citations:start -->
**Numbered literature:** [[4]](https://jseluis.github.io/silva-networks/paper/references/#ref-4), [[5]](https://jseluis.github.io/silva-networks/paper/references/#ref-5), [[6]](https://jseluis.github.io/silva-networks/paper/references/#ref-6), [[7]](https://jseluis.github.io/silva-networks/paper/references/#ref-7), [[8]](https://jseluis.github.io/silva-networks/paper/references/#ref-8), [[9]](https://jseluis.github.io/silva-networks/paper/references/#ref-9), [[22]](https://jseluis.github.io/silva-networks/paper/references/#ref-22), [[23]](https://jseluis.github.io/silva-networks/paper/references/#ref-23), [[35]](https://jseluis.github.io/silva-networks/paper/references/#ref-35), [[36]](https://jseluis.github.io/silva-networks/paper/references/#ref-36), [[37]](https://jseluis.github.io/silva-networks/paper/references/#ref-37), [[38]](https://jseluis.github.io/silva-networks/paper/references/#ref-38), [[39]](https://jseluis.github.io/silva-networks/paper/references/#ref-39), [[40]](https://jseluis.github.io/silva-networks/paper/references/#ref-40). Each number opens the complete citation and its primary external source.
<!-- silva-numbered-citations:end -->


In [1]:
from pathlib import Path
import importlib.util
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/jseluis/silva-networks.git"

def find_local_silva_root():
    candidates = [Path.cwd(), Path("/content/silva-networks")]
    root = Path.cwd()
    while root != root.parent:
        candidates.append(root)
        root = root.parent
    for candidate in candidates:
        if (candidate / "src" / "silva_networks").exists():
            return candidate
    return None

root = find_local_silva_root()
if root is not None:
    sys.path.insert(0, str(root / "src"))
elif IN_COLAB and importlib.util.find_spec("silva_networks") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", f"git+{REPO_URL}"])

import torch
from silva_networks import (
    SILVADEQConfig,
    SolverConfig,
    implicit_adjoint_solve,
    make_silva_translation_flow_batch,
    silva_deq,
    silva_endpoint_error,
    silva_projected_qp_layer,
    silva_euler_flow_block,
    silva_fixed_point_block,
    silva_jacobian_regularization_loss,
    silva_multiscale_deq_block,
    silva_deq_flow,
    silva_quadratic_optimization_layer,
)

torch.manual_seed(7)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

## Fixed Point and Adjoint

The tutorial fixed point becomes a package block:

$$
z^\star=\tanh(W_z z^\star+W_xx+b).
$$

The implicit adjoint solves

$$
(I-J_f^\top)u=g.
$$

The Jacobian penalty estimates $\|J_f\|_F^2$ with Hutchinson probes.

In [2]:
x = torch.randn(3, 4, device=device)
block = silva_fixed_point_block(
    in_dim=4,
    state_dim=6,
    config=SolverConfig(solver="anderson", max_iter=12, alpha=0.6, history=3),
).to(device)

result = block(x, return_result=True)
transition = lambda z: block.transition(z, x)
grad_output = torch.ones_like(result.z)
adjoint = implicit_adjoint_solve(transition, result.z, grad_output, max_iter=8, tol=1e-4)
penalty = silva_jacobian_regularization_loss(transition, result.z, samples=1, weight=1e-3)

{
    "fixed_point_shape": tuple(result.z.shape),
    "solver": result.solver,
    "iterations": result.iterations,
    "final_residual": result.residual,
    "adjoint_residual": adjoint.residual,
    "jacobian_penalty": float(penalty.detach().cpu()),
}

{'fixed_point_shape': (3, 6),
 'solver': 'anderson',
 'iterations': 12,
 'final_residual': 0.0018495226977393031,
 'adjoint_residual': 9.925881022354588e-05,
 'jacobian_penalty': 0.0015039056306704879}

## Multi-State, Neural ODE, and Optimization Bridges

A multiscale DEQ writes $s=(z^{(1)},z^{(2)})$ and solves
$s^\star=F_\theta(s^\star,x)$. The generic DEQ engine can also solve a
tuple state directly. The neural ODE bridge runs explicit Euler steps, while
the optimization bridge checks a fixed-point KKT step against an exact
quadratic solve and a package-native constrained simplex projection.

In [3]:
x_small = torch.randn(2, 3, device=device)

multi_block = silva_multiscale_deq_block(
    in_dim=3,
    low_dim=3,
    high_dim=2,
    config=SolverConfig(solver="anderson", max_iter=10, alpha=0.6, history=3),
).to(device)
multi_result = multi_block(x_small, return_result=True)

init_state = (torch.zeros(2, 3, device=device), torch.zeros(2, 2, device=device))

def coupled_transition(state):
    z_low, z_high = state
    z_low_next = torch.tanh(0.35 * z_low + x_small)
    z_high_next = torch.tanh(0.30 * z_high + z_low[:, :2])
    return z_low_next, z_high_next

engine_result = silva_deq(
    coupled_transition,
    init_state,
    config=SILVADEQConfig(forward_max_iter=10, alpha=0.7, history=3),
    return_result=True,
)

ode = silva_euler_flow_block(dim=3, steps=5, step_size=0.05).to(device)
terminal, trajectory = ode(x_small, return_trajectory=True)

opt_layer = silva_quadratic_optimization_layer(
    in_dim=3,
    state_dim=3,
    config=SolverConfig(solver="picard", max_iter=30, alpha=1.0),
).to(device)
z_opt = opt_layer(x_small)
z_exact = opt_layer.exact_solution(x_small)

simplex_layer = silva_projected_qp_layer(
    in_dim=3,
    state_dim=3,
    constraint="simplex",
    simplex_mass=1.0,
    config=SolverConfig(solver="picard", max_iter=30, alpha=1.0),
).to(device)
z_simplex = simplex_layer(x_small)

{
    "multiscale_shape": tuple(multi_result.z.shape),
    "tuple_state_shapes": [tuple(t.shape) for t in engine_result.state],
    "ode_trajectory_shape": tuple(trajectory.shape),
    "quadratic_solution_gap": float(torch.linalg.norm(z_opt - z_exact).detach().cpu()),
    "simplex_row_sums": [float(v) for v in z_simplex.sum(dim=-1).detach().cpu()],
    "simplex_min": float(z_simplex.min().detach().cpu()),
}

{'multiscale_shape': (2, 5),
 'tuple_state_shapes': [(2, 3), (2, 2)],
 'ode_trajectory_shape': (6, 2, 3),
 'quadratic_solution_gap': 1.4393417586688884e-06,
 'simplex_row_sums': [1.0, 1.0],
 'simplex_min': 0.08431783318519592}

## RAFT and DEQ-Flow Bridge

For optical flow, the state is a displacement field $u(p)$. The compact
package model builds all-pairs correlation, recurrently refines a flow update,
and solves the fixed point

$$
u^\star=T_\theta(u^\star,I_1,I_2).
$$

This compact case demonstrates the equilibrium optical-flow formulation rather than a full benchmark configuration.
training recipe.

In [4]:
flow_batch = make_silva_translation_flow_batch(
    batch_size=1,
    height=8,
    width=8,
    shift=(1.0, 0.0),
    device=device,
)
flow_model = silva_deq_flow(
    feature_dim=4,
    hidden_dim=8,
    corr_radius=1,
    config=SolverConfig(solver="anderson", max_iter=6, alpha=0.6, history=3),
).to(device)
flow_result = flow_model(flow_batch.image1, flow_batch.image2, return_result=True)
epe = silva_endpoint_error(flow_result.flow, flow_batch.flow, flow_batch.valid)

{
    "flow_shape": tuple(flow_result.flow.shape),
    "flow_residual": flow_result.solver_result.residual,
    "endpoint_error": float(epe.detach().cpu()),
}

{'flow_shape': (1, 2, 8, 8),
 'flow_residual': 0.19958461821079254,
 'endpoint_error': 0.9787138104438782}

## Reporting Checklist

| If the experiment used | Cite |
| --- | --- |
| fixed-point or DEQ layer | SILVA package, Deep Equilibrium Models, Deep Implicit Layers |
| implicit adjoint | Deep Implicit Layers and the solver used for the linear system |
| Euler bridge | Neural ODEs |
| multiscale state | Multiscale Deep Equilibrium Models |
| Jacobian penalty | Jacobian-regularized DEQs and Hutchinson trace estimation |
| optimization bridge | OptNet or differentiable convex optimization layers, with the SILVA scope noted |
| optical flow | RAFT, DEQ-Flow, and the dataset or benchmark used |

## From 09 Method Adaptation Atlas to a Custom SILVA Family

The construction in this notebook can be separated into the universal
conditioned-equilibrium contract

$$
z_0=I_\eta(x),\qquad
z^\star=T_\theta(z^\star,x),\qquad
\widehat y=Q_\psi(z^\star).
$$

For this topic:

| Part | Concrete interpretation |
| --- | --- |
| Equilibrium state | the tensor solved to equilibrium |
| Condition | the observed input or source tensor |
| Repeated computation | the state-preserving transition evaluated by the root solver |
| Required invariants | shape, device, dtype, finiteness, and differentiability |
| Replaceable components | initializer, source encoder, transition, readout, and solver |

The initializer and source path are evaluated outside or alongside the root
solve. Only the state-preserving transition is repeated. Replacing an internal
architecture does not change this equation, provided the transition still maps
the same state space into itself.


In [5]:
import torch as silva_extension_torch
from torch import nn as silva_extension_nn

from silva_networks import (
    SILVAConditionedEquilibrium,
    SILVAZeroInitializer,
    SolverConfig,
    validate_silva_transition,
)


class NotebookExtensionTransition(silva_extension_nn.Module):
    def __init__(self, condition_dim=2, state_dim=3):
        super().__init__()
        self.source = silva_extension_nn.Linear(condition_dim, state_dim)
        self.state_field = silva_extension_nn.Sequential(
            silva_extension_nn.Linear(state_dim, 2 * state_dim),
            silva_extension_nn.Tanh(),
            silva_extension_nn.Linear(2 * state_dim, state_dim),
        )

    def forward(self, state, condition):
        return silva_extension_torch.tanh(
            self.source(condition) + 0.15 * self.state_field(state)
        )


silva_extension_torch.manual_seed(610)
notebook_condition = silva_extension_torch.linspace(-1.0, 1.0, 8).reshape(4, 2)
notebook_state0 = silva_extension_torch.zeros(4, 3)
notebook_transition = NotebookExtensionTransition()

notebook_report = validate_silva_transition(
    notebook_transition,
    notebook_state0,
    notebook_condition,
)
assert notebook_report.valid

with silva_extension_torch.no_grad():
    notebook_reference_step = silva_extension_torch.tanh(
        notebook_transition.source(notebook_condition)
        + 0.15 * notebook_transition.state_field(notebook_state0)
    )
silva_extension_torch.testing.assert_close(
    notebook_transition(notebook_state0, notebook_condition),
    notebook_reference_step,
)

notebook_custom_model = SILVAConditionedEquilibrium(
    notebook_transition,
    SILVAZeroInitializer(3),
    readout=silva_extension_nn.Linear(3, 1),
    config=SolverConfig(
        solver="picard",
        max_iter=40,
        tol=1e-7,
        backward_mode="implicit",
        backward_solver="gmres",
        anderson_batch_dims=1,
    ),
)
notebook_custom_result = notebook_custom_model(
    notebook_condition,
    return_result=True,
)
assert notebook_custom_result.output.shape == (4, 1)
assert notebook_custom_result.solver_result.residual < 1e-5

notebook_custom_result.output.square().mean().backward()
assert all(
    parameter.grad is not None and silva_extension_torch.isfinite(parameter.grad).all()
    for parameter in notebook_custom_model.parameters()
)
print("custom transition:", notebook_report)
print("equilibrium residual:", notebook_custom_result.solver_result.residual)


custom transition: SILVATransitionReport(state_shape=(4, 3), output_shape=(4, 3), preserves_shape=True, preserves_device=True, preserves_dtype=True, finite=True, differentiable=True, parameter_count=54)
equilibrium residual: 5.960464477539063e-08


## Numerical Equivalence, Compact Reproduction, and Scale

Before training, compare one packaged transition with an independently written
update:

$$
e_{\mathrm{step}}
=\frac{\|T_\theta(z,x)-T_{\mathrm{ref}}(z,x)\|_2}
{\|T_{\mathrm{ref}}(z,x)\|_2+\varepsilon}.
$$

After solving, report the fixed-point residual separately:

$$
e_{\mathrm{fp}}
=\frac{\|T_\theta(z^\star,x)-z^\star\|_2}
{\|z^\star\|_2+\varepsilon}.
$$

For this notebook, a compact reproduction must declare and assert
**fixed-point residual and task error against a deterministic target**. A full experiment must additionally record the
source dataset version and split, preprocessing, architecture widths, solver
and optimizer schedules, random seeds, baseline configuration, checkpoints,
and every deviation from the cited protocol.

The principal scaling axes are **state width, batch size, and data volume**. Increase one axis at
a time, retain the compact deterministic case as a regression test, and record
task error, domain-specific residual, forward residual, backward linear
residual, memory use, and runtime independently.

### Extension Exercises

1. Replace one component from this notebook while preserving its state and
   domain invariants.
2. Write the replacement first as an independent reference function, then as
   a module, and assert one-step equivalence.
3. Compare two solver configurations on the identical trained transition.
4. Add a compact baseline and a predeclared metric threshold.
5. Create a full-scale configuration without weakening the compact tests.

The complete authoring protocol is documented in
[Extending SILVA](https://jseluis.github.io/silva-networks/learn/extending-silva/).


In [6]:
notebook_reproduction_record = {
    "notebook": '09_method_adaptation_atlas.ipynb',
    "state": 'the tensor solved to equilibrium',
    "condition": 'the observed input or source tensor',
    "transition": 'the state-preserving transition evaluated by the root solver',
    "invariants": 'shape, device, dtype, finiteness, and differentiability',
    "compact_metric": 'fixed-point residual and task error against a deterministic target',
    "scale_axis": 'state width, batch size, and data volume',
}
assert all(notebook_reproduction_record.values())
notebook_reproduction_record


{'notebook': '09_method_adaptation_atlas.ipynb',
 'state': 'the tensor solved to equilibrium',
 'condition': 'the observed input or source tensor',
 'transition': 'the state-preserving transition evaluated by the root solver',
 'invariants': 'shape, device, dtype, finiteness, and differentiability',
 'compact_metric': 'fixed-point residual and task error against a deterministic target',
 'scale_axis': 'state width, batch size, and data volume'}

## Where to Go Next

| Question | Page |
| --- | --- |
| How does each source method map into SILVA? | [Method Adaptation Atlas](https://jseluis.github.io/silva-networks/learn/method-adaptation-atlas/) |
| How are complete architecture families represented? | [Paper Family Adaptations](https://jseluis.github.io/silva-networks/learn/paper-family-adaptations/) |
| Where are the primary references collected? | [Paper and References](https://jseluis.github.io/silva-networks/paper/references/) |
